# Quality assurance report

This report checks the UrbanSound8K copy configured in config.json: file integrity against the SHA-256 manifest, duplicate detection, audio readability and technical ranges, and metadata consistency when the metadata CSV is present.

The cells print every result from the run. The recorded run scanned a full copy of the release; the repository itself carries fold1 inside p2/data/raw. Point dataset_dir at any copy of the release to repeat the checks; the manifest defines what should be there.

Checks performed:

1. **File integrity.** Size and SHA-256 against manifest/urbansound8k_manifest.json, for every file present under the dataset dir.
2. **Unexpected files.** Files on disk that the manifest does not list.
3. **Duplicates.** Byte-identical files, detected through the digests computed in check 1.
4. **Audio properties.** Readability, sample rate, channels, duration, WAV subtype per file.
5. **Metadata CSV.** Records, missing values, duplicate rows, valid ranges. Runs only when UrbanSound8K.csv is present; reported as skipped otherwise.

These checks detect technical problems. They do not verify that the class labels describe the audio correctly.

In [1]:

import sys
from pathlib import Path

# find the repository root: the folder holding manifest/urbansound8k_manifest.json
root = Path.cwd()
while not (root / "manifest" / "urbansound8k_manifest.json").is_file():
    if root == root.parent:
        raise FileNotFoundError("run this notebook from inside the repository")
    root = root.parent
sys.path.insert(0, str(root / "src"))


import pandas as pd
import soundfile as sf
from IPython.display import display

from p3 import config as cfg, integrity

config = cfg.load_config(root)
dataset_dir = cfg.dataset_dir(root, config)
manifest = integrity.load_manifest(cfg.manifest_path(root))

print("repository:", root.name)
print("dataset dir:", config["dataset_dir"])
print("manifest entries:", len(manifest["files"]))


repository: IDIG4120_DatasetProduct_P3
dataset dir: ../part_01/archive
manifest entries: 8733


## 1. File integrity

In [2]:
summary = integrity.verify_dataset(dataset_dir, manifest)

print("manifest entries:", summary["manifest_files"])
print("verified, size and sha256 match:", summary["verified"])
print("digest mismatches:", len(summary["mismatched"]), summary["mismatched"][:5])
print("absent, not under the dataset dir:", summary["absent"])
print("unexpected files:", len(summary["unexpected"]), summary["unexpected"][:5])


manifest entries: 8733
verified, size and sha256 match: 8733
digest mismatches: 0 []
absent, not under the dataset dir: 0
unexpected files: 0 []


A repository can legitimately ship a subset of the release, so absent files are reported, not treated as errors. A digest mismatch or an unexpected file is an integrity failure and fails `python run.py --verify-only` with exit code 3.

## 2. Duplicates

In [3]:
by_digest = {}
for rel, sha in summary["digests"].items():
    by_digest.setdefault(sha, []).append(rel)
duplicates = {sha: rels for sha, rels in by_digest.items() if len(rels) > 1}

print("files compared:", len(summary["digests"]))
print("byte-identical duplicate files:", len(duplicates))
for sha, rels in list(duplicates.items())[:5]:
    print(" ", sha[:16], rels)


files compared: 8733
byte-identical duplicate files: 0


## 3. Audio properties

The dataset page states excerpts of at most 4 s, with sample rate, bit depth and channels as uploaded. Measured durations can land slightly above 4 s because of WAV framing, so the range check uses a 4.1 s tolerance.

In [4]:
records, unreadable = [], []
for entry in manifest["files"]:
    rel = entry["path"]
    if not rel.endswith(".wav"):
        continue
    path = dataset_dir / rel
    if not path.is_file():
        continue
    try:
        info = sf.info(path)
    except Exception as exc:
        unreadable.append((rel, str(exc)))
        continue
    records.append({
        "file": rel,
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "frames": info.frames,
        "duration_s": info.duration,
        "subtype": info.subtype,
    })

audio = pd.DataFrame(records)
print("audio files inspected:", len(audio))
print("unreadable files:", len(unreadable))


audio files inspected: 8732
unreadable files: 0


In [5]:
invalid = audio[
    (audio["sample_rate"] <= 0)
    | (audio["channels"] <= 0)
    | (audio["frames"] <= 0)
    | (audio["duration_s"] <= 0)
    | (audio["duration_s"] > 4.1)
]
print("files outside the valid ranges:", len(invalid))
print()
print("sampling rates:")
display(audio["sample_rate"].value_counts().sort_index())
print("channel counts:")
display(audio["channels"].value_counts().sort_index())
print("WAV subtypes:")
display(audio["subtype"].value_counts())
print("duration in seconds:")
display(audio["duration_s"].describe())


files outside the valid ranges: 0

sampling rates:


sample_rate
8000        12
11024        7
11025       39
16000       45
22050       44
24000       82
32000        4
44100     5370
48000     2502
96000      610
192000      17
Name: count, dtype: int64

channel counts:


channels
1     739
2    7993
Name: count, dtype: int64

WAV subtypes:


subtype
PCM_16       5758
PCM_24       2753
FLOAT         169
PCM_U8         43
MS_ADPCM        8
IMA_ADPCM       1
Name: count, dtype: int64

duration in seconds:


count    8732.000000
mean        3.607522
std         0.974394
min         0.050000
25%         4.000000
50%         4.000000
75%         4.000000
max         4.036647
Name: duration_s, dtype: float64

## 4. Metadata CSV

In [6]:
csv_path = dataset_dir / "UrbanSound8K.csv"
csv_entry = next(f for f in manifest["files"] if f["path"] == "UrbanSound8K.csv")

if csv_path.is_file():
    integrity.verify_file(csv_path, csv_entry)
    meta = pd.read_csv(csv_path)
    print("verified against the manifest:", csv_entry["sha256"])
    print("metadata records:", len(meta))
    print("columns:", list(meta.columns))
    print("missing values per column:")
    display(meta.isna().sum())
    print("duplicate rows:", int(meta.duplicated().sum()))
    print("duplicate file names:", int(meta["slice_file_name"].duplicated().sum()))
    print("folds outside 1-10:", int((~meta["fold"].between(1, 10)).sum()))
    print("classIDs outside 0-9:", int((~meta["classID"].between(0, 9)).sum()))
    print("salience values outside 1-2:", int((~meta["salience"].isin([1, 2])).sum()))
    print("rows with end <= start:", int((meta["end"] <= meta["start"]).sum()))
    print()
    print("records per class:")
    display(meta["class"].value_counts())
    print("records per fold:")
    display(meta["fold"].value_counts().sort_index())
else:
    print("skipped: UrbanSound8K.csv is not present under the dataset dir")
    print("the manifest entry for the file is:", csv_entry)


verified against the manifest: b207f7a53f9e5eb02a5dd71d58fb8b66c88c24c0637b926dc72d9d400a6a2354
metadata records: 8732
columns: ['slice_file_name', 'fsID', 'start', 'end', 'salience', 'fold', 'classID', 'class']
missing values per column:


slice_file_name    0
fsID               0
start              0
end                0
salience           0
fold               0
classID            0
class              0
dtype: int64

duplicate rows: 0
duplicate file names: 0
folds outside 1-10: 0
classIDs outside 0-9: 0
salience values outside 1-2: 0
rows with end <= start: 0

records per class:


class
dog_bark            1000
children_playing    1000
air_conditioner     1000
street_music        1000
engine_idling       1000
jackhammer          1000
drilling            1000
siren                929
car_horn             429
gun_shot             374
Name: count, dtype: int64

records per fold:


fold
1     873
2     888
3     925
4     990
5     936
6     823
7     838
8     806
9     816
10    837
Name: count, dtype: int64

## 5. Summary

The cells above print every count from this run. Read them as the report: verified files, digest mismatches, unexpected files, byte-identical duplicates, unreadable files, files outside the valid ranges, and the metadata checks when the CSV was present. The checks detect technical problems with the files and the metadata; label correctness is out of scope for all of them.